# Session 22: Language models and building with them

Build a word-matching search over the eight Lakeside Mart policies, test it on twelve questions with a failures column, tune it, then try the sentence-transformers and small model cells in Colab.

**Student version.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Split the policies into passages
Read each file in `policies`, cut it into sentences and keep the file name with each. Store the list in `passages`.

In [ ]:
import os, re
import numpy as np
from collections import Counter

passages = []
# Your turn: for each file in policies, tidy the spaces, split after a full stop
# followed by a capital letter, and append (file name, sentence) to passages


In [ ]:
_missing = [v for v in ('passages',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert len(passages) == 59, 'Expected 59 passages: split after a full stop and a capital letter'
assert passages[28] == ('loyalty_programme.txt', 'Points expire after 12 months.'), 'Passage 28 should be the loyalty sentence about points expiring. Keep the files in sorted order and the file name with each sentence'
print('Task 1 correct')

## Task 2. Search by shared words
Build `retrieve(question)` that scores each passage by the words it shares with the question (ignore the filler words) and returns the best passage and its score.

In [ ]:
FILLER = {'the', 'a', 'an', 'of', 'to', 'in', 'is', 'are', 'do', 'does', 'how', 'what',
          'can', 'my', 'for', 'on', 'at', 'and', 'or', 'it', 'be', 'by', 'with', 'if',
          'we', 'you', 'from', 'when', 'which', 'will', 'i'}

def words(text):
    # Your turn: return the set of lower case words in text, without the filler words
    return set()

def retrieve(question):
    # Your turn: score every passage by the number of shared words,
    # then return (best passage, best score)
    return ('', ''), 0


In [ ]:
(name, sentence), s = retrieve('How many days do I have to return an item?')
assert name == 'returns_policy.txt', 'The returns policy should win this question. Check words() and that retrieve returns the passage with the most shared words'
print('Task 2 correct')

## Task 3. Build the test sheet
Write twelve test questions as a table `tests` with columns `question`, `policy` and `answer`. Each `answer` must be copied word for word from the policy file. Starter rows are given. Add the missing ones from the files yourself.

In [ ]:
import pandas as pd

tests = pd.DataFrame([
    ('How long do points last?', 'loyalty_programme.txt', 'Points expire after 12 months'),
    ('How many days do I have to return an item?', 'returns_policy.txt', 'within 7 days of purchase'),
    ('How long do mobile money refunds take?', 'returns_policy.txt', 'up to 3 working days'),
    ('Which towns does delivery not reach?', 'delivery.txt', 'Gulu, Mbale, Mbarara or Fort Portal'),
    ('What is the delivery fee within 5 kilometres?', 'delivery.txt', 'UGX 5,000 within 5 kilometres'),
    ('What percent discount do Gold customers get?', 'loyalty_programme.txt', 'Gold customers receive 5 percent off'),
    # Your turn: add six more rows. Choose questions on data privacy, food safety,
    # opening hours, supplier quality, mobile money and the loyalty rules.
    # Copy each answer from the policy file exactly.
], columns=['question', 'policy', 'answer'])
print(len(tests), 'questions')

In [ ]:
_missing = [v for v in ('tests',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert len(tests) == 12, 'tests needs twelve rows. Add six more questions with their policy file and answer'
assert tests.policy.nunique() >= 6, 'Cover at least six different policy files'
for _, t in tests.iterrows():
    text = ' '.join(open('policies/' + t.policy).read().split())
    assert t.answer in text, 'Answer not found word for word in ' + t.policy + ': ' + t.answer
print('Task 3 correct')

## Task 4. Score the simple method
Write `score(tests, finder)` that runs each question through a finder and returns a table with the policy it got, whether the right sentence was found, and a `failure` column.

In [ ]:
def score(tests, finder):
    rows = []
    for _, t in tests.iterrows():
        (name, sentence), s = finder(t.question)
        # Your turn: right = did we get the right policy? found = is the answer in the sentence?
        # Then choose a failure label: '' or 'wrong policy' or 'right policy, wrong sentence'
        pass
    return pd.DataFrame(rows, columns=['question', 'policy', 'got', 'answer_found', 'failure'])

r1 = score(tests, retrieve)
print((r1.policy == r1.got).sum(), 'of 12 right policy;', r1.answer_found.sum(), 'answers found')


In [ ]:
_missing = [v for v in ('r1',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert (r1.policy == r1.got).sum() >= 9, 'At least 9 of 12 should pick the right policy. Check your word sets.'
assert set(r1.failure) <= {'', 'wrong policy', 'right policy, wrong sentence'}
print('Task 4 correct')

## Task 5. Tune the method
Read the failures. Add a short list of words that mean the same, cut a trailing s, and weight rare words more. Build `retrieve2` and score it again. Your tuned method must pick the right policy for at least 11 of 12 questions and find at least 11 answers.

In [ ]:
from collections import Counter

SAME = {}   # Your turn: add pairs of words that mean the same, a question word on the left and the policy word on the right

def base(w):
    # Your turn: apply SAME, then cut a trailing s from words longer than 3 letters
    return w

def words2(text):
    return {base(w) for w in words(text)}

sets = [words2(p) for n, p in passages]
seen_in = Counter(w for s in sets for w in s)

def retrieve2(question):
    # Your turn: score each passage by adding a weight for every shared word.
    # A word that appears in fewer passages (see seen_in) should weigh more.
    return ('', ''), 0

r2 = score(tests, retrieve2)
print((r2.policy == r2.got).sum(), 'of 12 right policy;', r2.answer_found.sum(), 'answers found')


In [ ]:
_missing = [v for v in ('r1', 'r2',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert (r2.policy == r2.got).sum() >= 11, 'Fewer than 11 right policies. Look at the failure column for r2 and fix the cause'
assert r2.answer_found.sum() >= 11, 'Fewer than 11 answers found. Look at the failure column for r2'
assert (r2.policy == r2.got).sum() >= (r1.policy == r1.got).sum(), 'The tuned method should not do worse than the simple one'
print('Task 5 correct')

## Task 6. Search by meaning (Colab only, needs internet)
Run the sentence-transformers version of the search. This cell downloads a model of about 90 MB the first time. Score it on your test sheet. Your result will differ from the tuned word method.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
# Your turn: encode every passage once with normalize_embeddings=True
vectors = None   # replace None

def retrieve3(question):
    # Your turn: encode the question, take the dot product with vectors,
    # return the best passage and its score
    return ('', ''), 0

r3 = score(tests, retrieve3)
print((r3.policy == r3.got).sum(), 'of 12 right policy;', r3.answer_found.sum(), 'answers found')


In [ ]:
_missing = [v for v in ('r3',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
# r3 comes from the SentenceTransformer cell above
assert (r3.policy == r3.got).sum() >= 7, 'Fewer than 7 right policies. Encode every passage with normalize_embeddings=True, encode the question the same way, and take the best dot product'
print('Task 6 correct')

## Task 7. Let a small model answer (optional, Colab only)
Check the licence on the model card first. Then run the cell below. The model answers only from the passage that retrieval found. Compare three replies with the policy text.

In [ ]:
from transformers import pipeline

generate = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B-Instruct')

def answer(question):
    # Your turn: retrieve the best passage with retrieve2, send the model a rule
    # and the passage and question as messages, and return the reply text
    pass

print(answer('How long do points last?'))


## Task 8. Record and reflect
Write two lines below. First, one entry for your use log: what you asked an assistant, what it gave, how you checked, and whether you kept it (write 'none used' if you used no assistant). Second, one sentence on the failure you found hardest to fix, and why. Do not paste any customer data into an assistant.